In [ ]:
# Instalar PyTorch con CUDA (primero, para evitar que sentence-transformers instale CPU)
%pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
# Core libs
%pip install pandas==2.2.2 matplotlib==3.9.0 \
datasets==2.20.0 pyarrow==15.0.2 \
lime==0.2.0.1 shap==0.45.1

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

In [ ]:
import gc
import pandas as pd

from datasets import load_dataset
from datasets import Dataset

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)
from transformers import AutoConfig

In [ ]:
dataset = load_dataset("imdb")

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_MODEL_LENGTH = 2048
MAX_LENGTH = 2048

In [ ]:
def build_prompt(review):

    prompt = f"""Review: {review}

Sentiment:"""

    return prompt


def build_full_text(review, label):

    sentiment = (
        "positive"
        if label == 1
        else "negative"
    )

    full_text = f"""Review: {review}

Sentiment: {sentiment}"""

    return full_text


def preprocess_dataset(dataset, tokenizer):

    input_ids_list = []
    attention_masks_list = []
    labels_list = []
    true_labels_list = []

    for example in dataset:

        review = example["text"]
        label = example["label"]

        prompt = build_prompt(review)
        full_text = build_full_text(review, label)

        # =========================
        # TOKENIZACIÓN
        # =========================

        full = tokenizer(
            full_text,
            add_special_tokens=False
        )

        input_ids = full["input_ids"]

        # attention mask inicial
        attention_mask = [1] * len(input_ids)

        # =========================
        # LABELS (SIN prompt_len)
        # =========================

        labels = []

        # reconstruimos prompt tokenizado directamente dentro del full_text
        prompt_text = build_prompt(review)

        prompt_ids = tokenizer(
            prompt_text,
            add_special_tokens=False
        )["input_ids"]

        # ⚠️ no asumimos alineación por slicing, buscamos corte seguro por longitud
        prompt_len = len(prompt_ids)

        for i, tok in enumerate(input_ids):

            if i < prompt_len:
                labels.append(-100)
            else:
                labels.append(tok)

        # =========================
        # PADDING / TRUNCATION UNIFORME
        # =========================

        if len(input_ids) > MAX_LENGTH:

            input_ids = input_ids[:MAX_LENGTH]
            attention_mask = attention_mask[:MAX_LENGTH]
            labels = labels[:MAX_LENGTH]

        else:

            pad_len = MAX_LENGTH - len(input_ids)

            input_ids += [tokenizer.pad_token_id] * pad_len
            attention_mask += [0] * pad_len
            labels += [-100] * pad_len

        # =========================
        # TRUE LABEL EXPLÍCITO
        # =========================

        true_label = "positive" if label == 1 else "negative"

        # =========================
        # STORE
        # =========================

        input_ids_list.append(input_ids)
        attention_masks_list.append(attention_mask)
        labels_list.append(labels)
        true_labels_list.append(true_label)

    return Dataset.from_dict({
        "input_ids": input_ids_list,
        "attention_mask": attention_masks_list,
        "labels": labels_list,
        "true_label": true_labels_list
    })

In [ ]:
# Model tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def fits_model(example):

    text = build_full_text(
        example["text"],
        example["label"]
    )

    tokens = tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]

    return len(tokens) <= MAX_MODEL_LENGTH

In [ ]:
# Reviews exceeding model context are removed from the dataset.
BATCH_SIZE = 1

filtered_test = dataset["test"].filter(
    fits_model
)

test_dataset = preprocess_dataset(filtered_test, tokenizer)

test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels",
        "true_label"
    ]
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
teacher_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
    attn_implementation="eager"
)

teacher_model.load_state_dict(
    torch.load(
        "./checkpoints/best_teacher_model.pt",
        map_location=device
    )
)

# Métricas de comparación

In [ ]:
def frobenius_difference(
    x: torch.Tensor,
    y: torch.Tensor
):
    return torch.norm(
        x - y,
        p="fro"
    )


def cosine_similarity_tensor(
    x: torch.Tensor,
    y: torch.Tensor,
    eps: float = 1e-8
):
    x_flat = x.reshape(-1)
    y_flat = y.reshape(-1)

    similarity = F.cosine_similarity(
        x_flat.unsqueeze(0),
        y_flat.unsqueeze(0),
        dim=1,
        eps=eps
    )

    return similarity.squeeze()


def js_divergence_vectorized_attention(
    attn_teacher: torch.Tensor,
    attn_student: torch.Tensor,
    eps: float = 1e-8
):
    p = attn_teacher.reshape(-1)
    q = attn_student.reshape(-1)

    # normalizar como distribuciones
    p = p / (p.sum() + eps)
    q = q / (q.sum() + eps)

    m = 0.5 * (p + q)

    kl_pm = torch.sum(
        p * torch.log((p + eps) / (m + eps))
    )

    kl_qm = torch.sum(
        q * torch.log((q + eps) / (m + eps))
    )

    jsd = 0.5 * (kl_pm + kl_qm)

    return jsd


def js_divergence_rowwise_attention(
    attn_teacher: torch.Tensor,
    attn_student: torch.Tensor,
    eps: float = 1e-8
):
    # normalizar filas
    p = attn_teacher / (
        attn_teacher.sum(dim=-1, keepdim=True)
        + eps
    )

    q = attn_student / (
        attn_student.sum(dim=-1, keepdim=True)
        + eps
    )

    m = 0.5 * (p + q)

    kl_pm = torch.sum(
        p * torch.log(
            (p + eps) / (m + eps)
        ),
        dim=-1
    )

    kl_qm = torch.sum(
        q * torch.log(
            (q + eps) / (m + eps)
        ),
        dim=-1
    )

    jsd_rows = 0.5 * (
        kl_pm + kl_qm
    )

    return jsd_rows.mean()

# Rango efectivo

In [ ]:
def effective_rank_participation_ratio(
    x: torch.Tensor,
    eps: float = 1e-12
):

    x = x.float()

    singular_values = (
        torch.linalg.svdvals(x)
    )

    power = singular_values**2

    numerator = (
        power.sum()**2
    )

    denominator = (
        (power**2).sum()
        + eps
    )

    rank_eff = (
        numerator
        / denominator
    )

    return rank_eff


def effective_rank_entropy(
    x: torch.Tensor,
    eps: float = 1e-12
):

    x = x.float()

    singular_values = (
        torch.linalg.svdvals(x)
    )

    p = singular_values / (
        singular_values.sum()
        + eps
    )

    entropy = -torch.sum(
        p * torch.log(p + eps)
    )

    rank_eff = torch.exp(entropy)

    return rank_eff

# Atención promedio

In [ ]:
def compute_average_attentions(
    model,
    dataloader,
    device,
    max_batches=None,
    print_every=50
):

    model.eval()

    n_layers = (
        model.config.num_hidden_layers
    )

    attention_sums = [

        torch.zeros(
            (2048, 2048),
            dtype=torch.float32,
            device="cpu"
        )

        for _ in range(n_layers)
    ]

    n_examples = 0

    with torch.no_grad():

        for batch_idx, batch in enumerate(
            dataloader
        ):

            if (
                max_batches is not None
                and batch_idx >= max_batches
            ):
                break

            input_ids = batch[
                "input_ids"
            ].to(device)

            attention_mask = batch[
                "attention_mask"
            ].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_attentions=True
            )

            attentions = (
                outputs.attentions
            )

            batch_size = (
                input_ids.size(0)
            )

            for layer_idx in range(
                n_layers
            ):

                attn = attentions[
                    layer_idx
                ]

                # (B, H, S, S)
                # promedio heads
                attn = attn.mean(
                    dim=1
                )

                # (B, S, S)
                attn = (
                    attn
                    .float()
                    .cpu()
                )

                # suma batch
                attn_sum = attn.sum(
                    dim=0
                )

                attention_sums[
                    layer_idx
                ] += attn_sum

            n_examples += batch_size

            if (
                batch_idx
                % print_every
                == 0
            ):

                print(
                    f"Batch "
                    f"{batch_idx} | "
                    f"Examples "
                    f"{n_examples}"
                )

            del (
                input_ids,
                attention_mask,
                outputs,
                attentions,
                attn
            )

            gc.collect()

            torch.cuda.empty_cache()

    attention_means = [

        attn_sum / n_examples

        for attn_sum
        in attention_sums
    ]

    return attention_means

## Teacher

In [ ]:
teacher_model.to(device)
teacher_model.eval()

In [ ]:
teacher_model.config.output_attentions = True

In [ ]:
teacher_attentions = compute_average_attentions(teacher_model, test_loader, device)

In [ ]:
teacher_model.cpu()

In [ ]:
teacher_avg_attentions = [
    (teacher_attentions[2*i] + teacher_attentions[2*i + 1]) / 2
    for i in range(11)
]

In [ ]:
def rollout_pair(A1, A2):
    I = torch.eye(A1.shape[-1], device=A1.device)
    
    # agregar residual y renormalizar
    A1 = A1 + I
    A1 = A1 / A1.sum(dim=-1, keepdim=True)
    
    A2 = A2 + I
    A2 = A2 / A2.sum(dim=-1, keepdim=True)
    
    return A1 @ A2

teacher_rollouts = [rollout_pair(teacher_attentions[2*i], teacher_attentions[2*i+1]) for i in range(11)]

## Teacher - Student comparison

In [ ]:
# cargar config original
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)

# student de 11 capas
config.num_hidden_layers = 11

# crear arquitectura vacía
student_model = (
    AutoModelForCausalLM.from_config(
        config,
        attn_implementation="eager"
    )
)

# cargar pesos
student_model.load_state_dict(
    torch.load(
        "./checkpoints/best_student_model.pt",
        map_location=device
    )
)

student_model.to(device)
student_model.eval()

In [ ]:
student_model.config.output_attentions = True

In [ ]:
student_attentions = compute_average_attentions(student_model, test_loader, device)

In [ ]:
student_model.cpu()

In [ ]:
# Cosine similarity entre attention maps
print("Similitud coseno con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_avg_attentions)):
    print(cosine_similarity_tensor(sattn, tattn))

print("\n")

print("Similitud coseno con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_rollouts)):
    print(cosine_similarity_tensor(sattn, tattn))

In [ ]:
# Frobenius distance entre attention maps
print("Distancia Frobenius con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_avg_attentions)):
    print(frobenius_difference(sattn, tattn))

print("\n")

print("Distancia Frobenius con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_rollouts)):
    print(frobenius_difference(sattn, tattn))

In [ ]:
# JS global
print("JS global con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_avg_attentions)):
    print(js_divergence_vectorized_attention(sattn, tattn))

print("\n")

print("JS global con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_rollouts)):
    print(js_divergence_vectorized_attention(sattn, tattn))

In [ ]:
# JS local
print("JS promediada por fila con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_avg_attentions)):
    print(js_divergence_rowwise_attention(sattn, tattn))

print("\n")

print("JS promediada por fila con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_rollouts)):
    print(js_divergence_rowwise_attention(sattn, tattn))

In [ ]:
# Participation ratio
print("Participation ratio")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (sattn, tattn, tattn2) in enumerate(zip(student_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_participation_ratio(sattn)} | {effective_rank_participation_ratio(tattn)} | {effective_rank_participation_ratio(tattn2)}")


In [ ]:
# Entropy based effective rank
print("Entropy based effective rank")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (sattn, tattn, tattn2) in enumerate(zip(student_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_entropy(sattn)} | {effective_rank_entropy(tattn)} | {effective_rank_entropy(tattn2)}")

## Teacher - Student local comparison

In [ ]:
# crear arquitectura vacía
student_local_model = (
    AutoModelForCausalLM.from_config(
        config,
        attn_implementation="eager"
    )
)

# cargar pesos
student_local_model.load_state_dict(
    torch.load(
        "./checkpoints/best_student_local_model.pt",
        map_location=device
    )
)

student_local_model.to(device)
student_local_model.eval()

In [ ]:
student_local_model.config.output_attentions = True

In [ ]:
student_local_attentions = compute_average_attentions(student_local_model, test_loader, device)

In [ ]:
student_local_model.cpu()

In [ ]:
# Cosine similarity entre attention maps
print("Similitud coseno con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_avg_attentions)):
    print(cosine_similarity_tensor(sattn, tattn))

print("\n")

print("Similitud coseno con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_rollouts)):
    print(cosine_similarity_tensor(sattn, tattn))

In [ ]:
# Frobenius distance entre attention maps
print("Distancia Frobenius con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_avg_attentions)):
    print(frobenius_difference(sattn, tattn))

print("\n")

print("Distancia Frobenius con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_rollouts)):
    print(frobenius_difference(sattn, tattn))

In [ ]:
# JS global
print("JS global con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_avg_attentions)):
    print(js_divergence_vectorized_attention(sattn, tattn))

print("\n")

print("JS global con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_rollouts)):
    print(js_divergence_vectorized_attention(sattn, tattn))

In [ ]:
# JS local
print("JS promediada por fila con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_avg_attentions)):
    print(js_divergence_rowwise_attention(sattn, tattn))

print("\n")

print("JS promediada por fila con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_rollouts)):
    print(js_divergence_rowwise_attention(sattn, tattn))

In [ ]:
# Participation ratio
print("Participation ratio")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (sattn, tattn, tattn2) in enumerate(zip(student_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_participation_ratio(sattn)} | {effective_rank_participation_ratio(tattn)} | {effective_rank_participation_ratio(tattn2)}")


In [ ]:
# Entropy based effective rank
print("Entropy based effective rank")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (sattn, tattn, tattn2) in enumerate(zip(student_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_entropy(sattn)} | {effective_rank_entropy(tattn)} | {effective_rank_entropy(tattn2)}")